In [1]:
# numpy는 숫자 여러 개를 배열로 묶어 한 번에 계산하게 해 주는 라이브러리
# 이 노트북에서는 데이터 준비/정규화/학습 전 확인 같은 'NumPy 흐름'에 사용
import numpy as np

# torch는 PyTorch 라이브러리
# 이번 노트북에서는 (1) 자동 미분(autograd)과 (2) optimizer(torch.optim.SGD)를 쓰기 위해 사용
# - 자동 미분 : 사람이 미분 공식을 적지 않아도 PyTorch가 a.grad, b.grad를 계산해 주는 기능
# - optimizer: 사람이 직접 하던 a, b 업데이트를 PyTorch가 대신 수행해 주는 도구
import torch

In [2]:
# 1. 입력값 X와 정답 y 준비(이전 파일과 동일)

# X: 입력값으로, 여기서는 사람의 키(cm)를 사용
#    np.array([...])는 여러 숫자를 하나의 NumPy 배열로 묶는 것
X = np.array([160, 170, 180, 190])

# y: 정답값으로, 0은 농구선수 아님, 1은 농구선수임
#    x와 y는 순서대로 짝지어짐. 즉, 키 160cm -> 정답 0, 키 190cm -> 정답 1
y = np.array([0, 0, 1, 1])

print('입력값 X:', X)
print('정답값 y:', y)

입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [3]:
# 2. 입력값 정규화 (이전 파일과 동일)

# 평균과 표준편차를 계산
# - 평균: 데이터의 중심(가운데쯤 되는 값)
# - 표준편차: 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
# 입력값의 범위를 0 근처로 비슷하게 맞추면 학습이 더 안정적으로 진행됨
# 주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용
#       (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씀)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean:', X_mean)
print('입력값 표준편차 X_std:', X_std)
print('정규화된 입력값 X_norm:', X_norm)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.180339887498949
정규화된 입력값 X_norm: [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [5]:
# 2-1. X_norm과 y를 PyTorch tensor로 변환하고 shape을 (n, 1)로 정리 (이전 파일과 동일)

# dtype=torch.float32 : 소수점 계산(미분)을 위해 실수(float) 형식으로 만듦
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
# 즉 shape(n, 1)이어야 함. 그래서 reshape(-1, 1)로 모양을 바꿈
#   -1 : 행 개수는 알아서 (여기서는 4)
#    1 : 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print('학습용 입력 tensor X_norm_tensor:', X_norm_tensor)
print('학습용 정답 tensor y_tensor', y_tensor)

# shape을 꼭 확인! 둘 다 (4, 1)이어야 함
print('X_norm_tensor shape:', X_norm_tensor.shape)
print('y_tensor shape:', y_tensor.shape)

학습용 입력 tensor X_norm_tensor: tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape: torch.Size([4, 1])
y_tensor shape: torch.Size([4, 1])


In [ ]:
# 3. PerceptronModel 정의 (torch.nn.Module 상속)

# torch.nn.Module을 상속받아 우리만의 모델 class를 만듦
# 이전 파일에서 흩어져 있던 linear 생성과 H/z 계산을 이 안으로 모음
class PerceptronModel(torch.nn.Module):
    
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(1, 1)
    
    def forward(self, x):
        H = self.linear(x)
        
        z = torch.sigmoid(H)
        
        return z